# 04b Replenishment and Warehouse Allocation - Standardized Input

## Purpose

This notebook is the standardized-input version of `04_replenishment_warehouse_allocation.ipynb`. It reproduces the existing replenishment, inventory-risk, warehouse-allocation, and management-KPI logic using the SKU classification output created by `03b_sku_classification_standardized_input.ipynb`.

This is a preparation layer for future reusable pipeline refactoring. It does not replace the original workflow or change the validated project metrics.

## Input, output, and SKU scope

The notebook consumes:

```text
outputs_standardized/sku_profile_classification.csv
```

It writes four management outputs only to `outputs_standardized/`. The original `outputs/` and `data/processed/` directories are not modified.

`stock_code` remains the normalized SKU key. `description` is retained only as a display field and is not used as a grouping key.

## Important simulation notice

The source data does not contain real inventory balances, supplier lead times, unit costs, storage volumes, warehouse capacity, or fulfillment assignments. All inventory-related fields and recommendations in this notebook are deterministic simulations for portfolio demonstration only.

They are not real company inventory data and should not be used as operational, purchasing, accounting, or commercial decisions.

## Load the standardized SKU profile

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Resolve paths whether the notebook runs from the project root or notebooks/.
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
outputs_dir = project_root / "outputs_standardized"
sku_profile_path = outputs_dir / "sku_profile_classification.csv"

if not sku_profile_path.exists():
    raise FileNotFoundError(
        "Standardized SKU profile not found. Run "
        "notebooks/03b_sku_classification_standardized_input.ipynb first. "
        f"Expected file: {sku_profile_path}"
    )

sku_profile = pd.read_csv(
    sku_profile_path,
    dtype={"stock_code": "string", "description": "string"},
)

required_columns = {
    "stock_code", "description", "sku_class", "total_units",
    "total_revenue", "avg_monthly_units", "avg_unit_price", "demand_cv",
}
missing_columns = sorted(required_columns - set(sku_profile.columns))
if missing_columns:
    raise ValueError(
        "Missing standardized SKU profile columns: " + ", ".join(missing_columns)
    )
if sku_profile["stock_code"].duplicated().any():
    raise ValueError("SKU profile must contain one row per stock_code.")

outputs_dir.mkdir(parents=True, exist_ok=True)

print("SKU profile shape:", sku_profile.shape)
print("Unique stock_code values:", sku_profile["stock_code"].nunique())
sku_profile.head()

## Simulated inventory assumptions

The notebook uses the same random seed and simulated distributions as notebook 04:

- Current inventory depends on SKU class.
- Supplier lead time is drawn from 7, 14, 21, 30, or 45 days with the original probabilities.
- Storage volume per unit is drawn uniformly from 0.1 to 3.0.
- Unit cost is estimated as 35% to 65% of average unit price.

The standardized field `supplier_lead_time_days` represents the same simulated lead-time assumption used in the original model.

In [ ]:
np.random.seed(42)

sku_profile["current_inventory"] = np.where(
    sku_profile["sku_class"].isin(["High-Revenue Priority", "High-Turnover Stable"]),
    np.random.randint(20, 300, size=len(sku_profile)),
    np.random.randint(0, 120, size=len(sku_profile)),
)

sku_profile["supplier_lead_time_days"] = np.random.choice(
    [7, 14, 21, 30, 45],
    size=len(sku_profile),
    p=[0.20, 0.35, 0.25, 0.15, 0.05],
)

sku_profile["storage_volume_per_unit"] = np.random.uniform(
    0.1, 3.0,
    size=len(sku_profile),
).round(2)

sku_profile["unit_cost"] = (
    sku_profile["avg_unit_price"]
    * np.random.uniform(0.35, 0.65, size=len(sku_profile))
).round(2)

sku_profile[
    [
        "stock_code", "description", "sku_class", "current_inventory",
        "supplier_lead_time_days", "storage_volume_per_unit", "unit_cost",
    ]
].head()

## Reorder point and safety stock

Average monthly demand is converted to daily demand using 30 days. Safety stock, reorder point, and recommended replenishment quantity use the same formulas and rounding as notebook 04.

In [ ]:
# Convert monthly demand to daily demand.
sku_profile["avg_daily_demand"] = sku_profile["avg_monthly_units"] / 30

# Safety stock based on lead time and clipped demand volatility.
sku_profile["safety_stock"] = (
    sku_profile["avg_daily_demand"]
    * sku_profile["supplier_lead_time_days"]
    * (0.25 + sku_profile["demand_cv"].clip(0, 2) * 0.25)
).round(0)

# Reorder point.
sku_profile["reorder_point"] = (
    sku_profile["avg_daily_demand"] * sku_profile["supplier_lead_time_days"]
    + sku_profile["safety_stock"]
).round(0)

# Recommended replenishment quantity.
sku_profile["recommended_replenishment_qty"] = (
    sku_profile["reorder_point"] - sku_profile["current_inventory"]
).clip(lower=0).round(0)

sku_profile[
    [
        "stock_code", "description", "sku_class", "avg_monthly_units",
        "current_inventory", "supplier_lead_time_days", "safety_stock",
        "reorder_point", "recommended_replenishment_qty",
    ]
].head()

## Inventory risk

Risk rules preserve notebook 04's precedence:

1. Stockout Risk when current inventory is below the reorder point.
2. Overstock Risk when coverage exceeds 180 days for Long-Tail or Regular SKUs.
3. Normal for all remaining SKUs.

In [ ]:
sku_profile["inventory_coverage_days"] = np.where(
    sku_profile["avg_daily_demand"] > 0,
    sku_profile["current_inventory"] / sku_profile["avg_daily_demand"],
    np.inf,
)

def assign_inventory_risk(row):
    if row["current_inventory"] < row["reorder_point"]:
        return "Stockout Risk"
    elif (
        row["inventory_coverage_days"] > 180
        and row["sku_class"] in ["Long-Tail", "Regular"]
    ):
        return "Overstock Risk"
    else:
        return "Normal"

sku_profile["inventory_risk"] = sku_profile.apply(assign_inventory_risk, axis=1)

sku_profile[
    [
        "stock_code", "description", "sku_class", "current_inventory",
        "reorder_point", "inventory_coverage_days", "inventory_risk",
    ]
].head(20)

## Warehouse allocation strategy

Warehouse strategies use the same categories and rule order as notebook 04. Overstock review takes precedence, followed by the SKU-class strategies.

In [ ]:
def assign_warehouse_strategy(row):
    if row["inventory_risk"] == "Overstock Risk":
        return "Overstock Review / Reduce Replenishment"
    elif row["sku_class"] == "High-Revenue Priority":
        return "Local Warehouse Priority"
    elif row["sku_class"] == "High-Turnover Stable":
        return "Stable Local Warehouse Inventory"
    elif row["sku_class"] == "High-Turnover Volatile":
        return "Small-Batch Replenishment / Monitor Closely"
    elif row["sku_class"] == "Long-Tail":
        return "External or Limited Stock Strategy"
    else:
        return "Standard Replenishment Review"

sku_profile["warehouse_strategy"] = sku_profile.apply(
    assign_warehouse_strategy,
    axis=1,
)

sku_profile[
    ["stock_code", "description", "sku_class", "inventory_risk", "warehouse_strategy"]
].head(20)

## Management output tables

The notebook creates replenishment recommendations, an overstock-risk list, and a warehouse-allocation summary using the same filters, ordering, fields, and aggregations as notebook 04.

In [ ]:
replenishment_recommendations = (
    sku_profile[sku_profile["recommended_replenishment_qty"] > 0]
    .sort_values(
        ["sku_class", "recommended_replenishment_qty"],
        ascending=[True, False],
    )[
        [
            "stock_code",
            "description",
            "sku_class",
            "avg_monthly_units",
            "current_inventory",
            "supplier_lead_time_days",
            "safety_stock",
            "reorder_point",
            "recommended_replenishment_qty",
            "inventory_risk",
            "warehouse_strategy",
        ]
    ]
)

overstock_risk_list = (
    sku_profile[sku_profile["inventory_risk"] == "Overstock Risk"]
    .sort_values("inventory_coverage_days", ascending=False)[
        [
            "stock_code",
            "description",
            "sku_class",
            "avg_monthly_units",
            "current_inventory",
            "inventory_coverage_days",
            "inventory_risk",
            "warehouse_strategy",
        ]
    ]
)

warehouse_allocation_summary = (
    sku_profile
    .groupby(["sku_class", "warehouse_strategy"], as_index=False)
    .agg(
        sku_count=("stock_code", "count"),
        total_revenue=("total_revenue", "sum"),
        total_units=("total_units", "sum"),
        avg_inventory_coverage_days=("inventory_coverage_days", "mean"),
    )
    .sort_values("total_revenue", ascending=False)
)

replenishment_recommendations.head()

## Management KPI summary

The KPI table preserves notebook 04's eleven management metrics, including the current validated cleaning counts, SKU counts, inventory-risk counts, replenishment count, and warehouse-strategy group count.

In [ ]:
management_kpi_summary = pd.DataFrame({
    "metric": [
        "valid_product_sales_rows",
        "returns_cancellations_rows",
        "non_product_rows_excluded",
        "sku_master_count",
        "sku_profile_count",
        "high_revenue_priority_sku_count",
        "long_tail_sku_count",
        "stockout_risk_sku_count",
        "overstock_risk_sku_count",
        "replenishment_recommendation_count",
        "warehouse_strategy_group_count",
    ],
    "value": [
        522716,
        10587,
        2162,
        3917,
        len(sku_profile),
        (sku_profile["sku_class"] == "High-Revenue Priority").sum(),
        (sku_profile["sku_class"] == "Long-Tail").sum(),
        (sku_profile["inventory_risk"] == "Stockout Risk").sum(),
        (sku_profile["inventory_risk"] == "Overstock Risk").sum(),
        len(replenishment_recommendations),
        len(warehouse_allocation_summary),
    ],
})

management_kpi_summary

## Save standardized outputs

All four outputs are written only to `outputs_standardized/`.

In [ ]:
output_paths = {
    "replenishment_recommendations.csv": replenishment_recommendations,
    "overstock_risk_list.csv": overstock_risk_list,
    "warehouse_allocation_summary.csv": warehouse_allocation_summary,
    "management_kpi_summary.csv": management_kpi_summary,
}

for filename, output_table in output_paths.items():
    output_table.to_csv(outputs_dir / filename, index=False)

print("Saved output files:")
for filename in output_paths:
    print(f"- outputs_standardized/{filename}")

## Management interpretation

For the current project data and deterministic simulated assumptions, the model should identify 1,432 Stockout Risk SKUs, 924 Overstock Risk SKUs, and 1,561 SKUs in a Normal inventory position.

These results demonstrate the workflow but remain simulated. They do not represent real inventory recommendations or warehouse decisions.

## Final summary

The final cell lists output files and shapes, followed by the inventory-risk and warehouse-strategy distributions.

In [ ]:
print("===== 04b Standardized-Input Replenishment and Warehouse Allocation Summary =====")
print("\nOutput files and shapes:")
for filename, output_table in output_paths.items():
    print(f"- outputs_standardized/{filename}: {output_table.shape}")

print("\nInventory risk distribution:")
print(sku_profile["inventory_risk"].value_counts())

print("\nWarehouse strategy distribution:")
print(sku_profile["warehouse_strategy"].value_counts())